# University Project: Large Language Models from Scratch
**Group:** Phillip Graf, Konstantin Schmidt, Fabian Holländer

## Lab 2: Tokenization and positional embeddings
In this lab, we import our dataset and perform the necessary preprocessing steps. We then focus on generating tokens from the recipe data and, as a final step, implement positional embeddings to prepare the sequences for the model.

For learning and testing purposes we import a reduced dataset of 100.000 reciepes.
https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv

*Optional* the full dataset containing 2.2 million recipes can be used as input for larger-scale training.
https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/full_dataset.csv

## Import necessary libaries

In [2]:
import pandas as pd
import re
import importlib
import tiktoken
import torch

## Import the dataset from the cloud

In [3]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
# Uncomment the following line to use the full dataset
# cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/full_dataset.csv"

try:
    print("Loading dataset from cloud...")
    df = pd.read_csv(cloud_url)
    print("Dataset loaded successfully!\n")
    print("Info")
    print(df.info())
    print("")
    print("Head of the dataset:")
    print(df.head())
    
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

Loading dataset from cloud...
Dataset loaded successfully!

Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Unnamed: 0   100000 non-null  int64 
 1   title        100000 non-null  object
 2   ingredients  100000 non-null  object
 3   directions   100000 non-null  object
 4   link         100000 non-null  object
 5   source       100000 non-null  object
 6   NER          100000 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
None

Head of the dataset:
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar c

## Formatting of the dataset to one input string
This allows us to deduplicate the vocabulary, generate consistent tokens, and significantly streamline sequence iteration later in the process.

Since only the three columns **title**, **ingredients**, and **directions** are necessary for text understanding, we use only these to generate our concatenated input string.

In [4]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients: {ingredients}\nDirections: {directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"The entire dataset has been successfully converted into a single string.")
print(f"Total character count: {len(raw_text)}")
print("\nPreview of the first 100 characters:")
print(raw_text[:100])

# TODO -> umstellen auf full raw text?
# For a faster computation we only select the first 100 characters of the text for the next steps
raw_text = raw_text[:100]

The entire dataset has been successfully converted into a single string.
Total character count: 47074254

Preview of the first 100 characters:
Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/


## Building a simple tokenizer

In this step, we develop a straightforward tokenizer to process the concatenated dataset:
- The input string is split at spaces and specific punctuation marks that serve as word separators, including: , . : ; ? _ ! " ( )
- This process results in a list of tokens. In this context, a token can be a complete word, a single character, or a sequence of special symbols (e.g., @#).
---

**Example**

```This is a sample text: with numbers 123, special characters !@# and a few more. .```

The tokenizer identifies individual words like "text", but also preserves concatenated special characters such as "@#" as distinct tokens.


In [5]:
# the tokenizer function (basically just a split function)
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens

In [6]:
# Example
example_text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
result = split_text(example_text)
print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


In [7]:
# Tokenizer on our dataset
split = split_text(raw_text)
print(f"Anzahl der Wörter: {len(split)}")
print(split)

Anzahl der Wörter: 22
['Recepie', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk', ',', '1/']


## Converting Tokens into Token IDs
This is a crucial preprocessing step for the neural network, as the model can only process numerical values.
In the following cell, you can see a function and its output demonstrating how to map a token to an ID. The process is straightforward: the first token in the list is assigned ID 1, the second token is assigned ID 2, and so on.

This outputs the **vocabulary** (vocab), a list of all token ids mapped with the tokens of the dataset.

In [8]:
all_words = set(split)
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('1/2', 0)
('sugar', 1)
('Cookies', 2)
('1/', 3)
(':', 4)
('brown', 5)
('packed', 6)
('Nut', 7)
('Ingredients', 8)
('firmly', 9)
('.', 10)
(',', 11)
('evaporated', 12)
('1', 13)
('milk', 14)
('No-Bake', 15)
('c', 16)
('Recepie', 17)


## Building a whole tokenizer system
The cell below combines these components into a Tokenizer class. This class takes text as input, splits it into tokens, and maps those tokens to IDs, as we saw in the cells above. Additionally, it includes a decoding function to convert token IDs back into strings. This decoding step is essential for us to translate the model's numerical output back into human-readable text.

In [9]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [10]:
# Example on our raw_text
tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode(raw_text)
print(f"Input text: {raw_text}\n")
print(f"Token IDs: {ids}")

decoded_ids = tokenizer.decode(ids)
print(f"\nToken IDs decoded to text: {decoded_ids}")

Input text: Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/

Token IDs: [17, 4, 15, 7, 2, 8, 4, 13, 16, 10, 9, 6, 5, 1, 11, 0, 16, 10, 12, 14, 11, 3]

Token IDs decoded to text: Recepie : No-Bake Nut Cookies Ingredients : 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/


## Add specials tokens
In the next step, we integrate special tokens into our tokenizer to handle specific structural and vocabulary requirements:
- <|unk|> (Unknown Token): This token represents words or symbols that were not present in the training vocabulary. Instead of throwing an error when encountering an unseen word during encoding, the system simply maps it to the <|unk|> ID
- <|endoftext|> (End of Text Token): This serves as a signal to the model that the current sentence or phrase has ended.
---
**Example**

With the vocabulary of our (reduced) dataset (see above) and the following input example:

``Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.``

We get this encoded list of token IDs:

``[19, 0, 19, 19, 19, 19, 19, 18, 19, 19, 19, 19, 19, 19, 19, 1]``

When we decode these token IDs, we get the following tokens in return:

``<|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|endoftext|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>.``

This means that all of these words are unfortunaly unknown to our vocabulary. Only the punctuation marks ',' and '.' are recognized as known tokens.

In [11]:
#adjusted tokenizer
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [12]:
# add the new special tokens to the vocabulary
all_tokens = sorted(list(set(split)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [13]:
# Test the new tokenizer with the updated vocabulary on an example input text
tokenizer = SimpleTokenizerV2(vocab)

# Example sentences
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(f"Example text: \n{text}\n")

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}\n")

decoded_tokens = tokenizer.decode(token_ids)
print(f"Decoded text: \n{decoded_tokens}\n")

Example text: 
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

Token IDs: [19, 0, 19, 19, 19, 19, 19, 18, 19, 19, 19, 19, 19, 19, 19, 1]

Decoded text: 
<|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|endoftext|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>.



## Byte-pair-encoding

Our current tokenizer only recognizes exact words. By breaking unfamiliar terms into sub-tokens instead of simply labeling them as "unknown," we ensure the model can still process new words as long as their individual components are part of the vocabulary.

Instead of building this by our own, we use OpenAI’s tiktoken library with the cl100k_base encoding. Compared to the older gpt2 encoding (used as an example in the original chapters), it offers a larger vocabulary, a superior text compression and better support for special characters.

---
**Example**

The example below demonstrates the impact of Byte-Pair Encoding (BPE) compared to our SimpleTokenizer. Observe how the token count changes when spaces are removed.

Standard text:

``Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.``

Compressed text without white spaces:

``Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace.``

- SimpleTokenizerV2:
    - Standard text: Generates 16 tokens.
    - Compressed text (no spaces): Generates only 6 tokens. (It fails to recognize individual words without spaces).

- Tiktoken (OpenAI):
    - Standard text: Generates 19 tokens.
    - Compressed text (no spaces): Generates 22 tokens. (It identifies sub-word units, maintaining the semantic meaning even without whitespace).

In [14]:
tokenizer = SimpleTokenizerV2(vocab)

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.")

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}\n")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"Decoded text: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)

Token IDs: [19, 0, 19, 19, 19, 19, 19, 18, 19, 19, 19, 19, 19, 19, 19, 1]

Token Count: 16
Decoded text: 
<|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|endoftext|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>.

List of individual tokens:
['<|unk|>', ',', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|endoftext|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '<|unk|>', '.']


In [15]:
text = ( "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace." )

# Get the token IDs for the example text
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}\n")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"Decoded text: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)


Token IDs: [19, 0, 19, 19, 19, 1]

Token Count: 6
Decoded text: 
<|unk|>, <|unk|> <|unk|> <|unk|>.

List of individual tokens:
['<|unk|>', ',', '<|unk|>', '<|unk|>', '<|unk|>', '.']


In [16]:
tokenizer = tiktoken.get_encoding("cl100k_base")

# Example text with the byte-pair-encoding
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.")
token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(f"Token IDs: {token_ids}")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"\nDecoded Tokens: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)


Token IDs: [9906, 11, 656, 499, 1093, 15600, 30, 220, 100257, 763, 279, 7160, 32735, 7317, 2492, 315, 279, 44439, 13]
Token Count: 19

Decoded Tokens: 
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

List of individual tokens:
['Hello', ',', ' do', ' you', ' like', ' tea', '?', ' ', '<|endoftext|>', ' In', ' the', ' sun', 'lit', ' terr', 'aces', ' of', ' the', ' palace', '.']


In [17]:
# Example text but without spaces to see the effect of the byte-pair-encoding
text = ( "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace." )
token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(f"Token IDs: {token_ids}")
print(f"Token Count: {len(token_ids)}")

decoded_tokens = tokenizer.decode(token_ids)
print(f"\nDecoded Tokens: \n{decoded_tokens}\n")

# Decode tokens individually into a list without repr()
decoded_list = [tokenizer.decode([t_id]) for t_id in token_ids]

# Print the list
print("List of individual tokens:")
print(decoded_list)

Token IDs: [9906, 12260, 2303, 283, 7792, 7870, 64, 30, 100257, 644, 6509, 359, 75, 3328, 81, 2492, 1073, 339, 752, 278, 580, 13]
Token Count: 22

Decoded Tokens: 
Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace.

List of individual tokens:
['Hello', ',d', 'oy', 'ou', 'lik', 'ete', 'a', '?', '<|endoftext|>', 'In', 'thes', 'un', 'l', 'itter', 'r', 'aces', 'of', 'th', 'ep', 'al', 'ace', '.']


In [18]:
# Generate a comparison table for the two tokenizers on the two example texts

# Initialize tokenizers
tokenizer_v2 = SimpleTokenizerV2(vocab)
tokenizer_tik = tiktoken.get_encoding("cl100k_base")

# Define input strings
text_standard = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
text_compressed = "Hello,doyouliketea?<|endoftext|>Inthesunlitterracesofthepalace."

results = []

# Process strings
for label, text in [("Standard text (with spaces)", text_standard), ("Compressed text (no spaces)", text_compressed)]:
    # SimpleTokenizerV2
    ids_v2 = tokenizer_v2.encode(text)
    
    # Tiktoken (BPE)
    ids_tik = tokenizer_tik.encode(text, allowed_special={"<|endoftext|>"})
    
    results.append({
        "label": label,
        "v2_count": len(ids_v2),
        "tik_count": len(ids_tik)
    })

# Print comparison table
print(f"{'Text Version':<30} | {'SimpleV2 Count':<15} | {'Tiktoken Count':<15}")
print("-" * 60)
for res in results:
    print(f"{res['label']:<30} | {res['v2_count']:<15} | {res['tik_count']:<15}")

Text Version                   | SimpleV2 Count  | Tiktoken Count 
------------------------------------------------------------
Standard text (with spaces)    | 16              | 19             
Compressed text (no spaces)    | 6               | 22             


## Data Sampling with a Sliding Window

Once the text is tokenized, we need a mechanism to feed it into the neural network. Since a model learns to predict tokens sequentially, we train it to look at a sequence of previous words to predict the next one. To prevent the model from "cheating," it should only see the context preceding the target token.

To implement this, we use the Sliding Window approach:
- **Input Sequence:** A fixed-length window of tokens from the text.
- **Target Sequence:** The exact same window, but shifted by one position to the right.

### DataLoader
To efficiently feed our text into the model, we wrap the sliding window logic into a PyTorch Dataset and DataLoader. This allows us to handle large amounts of data, shuffle our sequences, and organize them into batches.


**Key Parameters**

``max_length``: The size of the "Context Window." It determines how many tokens the model looks at to predict the next one.

``stride``: The distance the window moves for the next sample. A stride smaller than max_length creates overlapping sequences, providing more training data and helps the model see words in various positions.

``batch_size``: The number of sequences processed simultaneously in one training step.

``shuffle``: When True, it randomizes the order of the sequences to prevent the model from memorizing the order of the source text.

---

**Example:**

The example below demonstrates how the DataLoader structures the data into Inputs (what the model sees) and Targets (what the model must predict). The targets are simply the inputs shifted by one position.

Setup:
- ``batch_size=1``
- ``max_length=4``
- ``stride=1``

Input Text:
``Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/``

DataLoader Output First Batch:
``[tensor([[  697,   346, 20898,    25]]), tensor([[  346, 20898,    25,  2360]])]``

Explanation:

``tensor([[  697,   346, 20898,    25]])`` -> Input Tensor  
(Corresponds to: ['Re', 'ce', 'pie', ':'] -> "Recepie:")

``tensor([[  346, 20898,    25,  2360]])]`` -> Target Tensor  
(Corresponds to: ['ce', 'pie', ':', ' No'] -> "cepie: No")

In [19]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [20]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("cl100k_base")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [21]:
def decode_batch(batch, tokenizer):
    inputs, targets = batch
    
    # Extract IDs for the first sequence in the batch
    input_ids = inputs[0].tolist()
    target_ids = targets[0].tolist()
    
    # Decode the full text
    input_text = tokenizer.decode(input_ids)
    target_text = tokenizer.decode(target_ids)
    
    # Decode tokens individually to see the fragments
    input_tokens = [tokenizer.decode([t_id]) for t_id in input_ids]
    target_tokens = [tokenizer.decode([t_id]) for t_id in target_ids]

    print(f"Input batch (Full):   {input_text}")
    print(f"Input tokens (List):   {input_tokens}")
    print("-" * 30)
    print(f"Target batch (Full):  {target_text}")
    print(f"Target tokens (List):  {target_tokens}")

In [22]:
# get the tokenized text
tokenizer = tiktoken.get_encoding("cl100k_base")
enc_text = tokenizer.encode(raw_text)
print(f"Raw text: {raw_text}")
print(f"\nEncoded text: {enc_text}")
print(f"\nLength of encoded text: {len(enc_text)}")

Raw text: Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/

Encoded text: [697, 346, 20898, 25, 2360, 7826, 731, 18878, 27085, 198, 46847, 25, 220, 16, 272, 13, 32620, 19937, 14198, 13465, 11, 220, 16, 14, 17, 272, 13, 60150, 660, 14403, 11, 220, 16, 14]

Length of encoded text: 34


In [23]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

# Text verfication
decode_batch(first_batch, tokenizer)

[tensor([[  697,   346, 20898,    25]]), tensor([[  346, 20898,    25,  2360]])]
Input batch (Full):   Recepie:
Input tokens (List):   ['Re', 'ce', 'pie', ':']
------------------------------
Target batch (Full):  cepie: No
Target tokens (List):  ['ce', 'pie', ':', ' No']


In [24]:
second_batch = next(data_iter)
print(second_batch)

# Text verfication
decode_batch(second_batch, tokenizer)

[tensor([[  346, 20898,    25,  2360]]), tensor([[20898,    25,  2360,  7826]])]
Input batch (Full):   cepie: No
Input tokens (List):   ['ce', 'pie', ':', ' No']
------------------------------
Target batch (Full):  pie: No-B
Target tokens (List):  ['pie', ':', ' No', '-B']


In [25]:
#Quick Example how the dataloader works with batches
dataloader = create_dataloader_v1(raw_text, batch_size=3, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  697,   346, 20898,    25],
        [ 2360,  7826,   731, 18878],
        [27085,   198, 46847,    25]])

Targets:
 tensor([[  346, 20898,    25,  2360],
        [ 7826,   731, 18878, 27085],
        [  198, 46847,    25,   220]])


## Token embeddings

Token Embeddings transform discrete integer IDs into continuous vectors of fixed size. Instead of treating words as arbitrary numbers, we represent them in a high-dimensional space where the model can learn relationships between them.

An embedding layer is essentially a lookup table. Instead of performing expensive matrix multiplications with one-hot encoded vectors, it directly retrieves the corresponding row from a weight matrix. Over time, the model adjusts these vectors so that words used in similar contexts end up closer together in the vector space.

### Understandings for ourself

**Why not just use IDs?**

While a tokenizer assigns a unique ID (integer) to every word, using these IDs directly for training is problematic:
- Arbitrary Distance: Is token 500 "closer" in meaning to 501 than to 10? No. In integer form, the values are arbitrary and convey no semantic relationship.
- Linear Scaling: A model would treat ID 1000 as "100 times more significant" than ID 10, which makes no sense for language.

Instead we are using embeddings.

**What are Embeddings?**

An Embedding replaces a single ID with a vector (a list of numbers).
Instead of one number, we use multiple dimensions (e.g., 256, 768, or 1536) to describe a token. Each dimension can theoretically learn a different feature of the word—such as its grammatical role, sentiment, or relationship to other concepts. This allows the model to calculate similarity: words like "cat" and "dog" will eventually have similar vectors and sit close together in this high-dimensional space.

These Embeddings are combined in an big Embedding Matrix.

**Embedding Matrix**

The Embedding Matrix is essentially a giant lookup table.
- Rows: The number of rows equals the Vocabulary Size (e.g., 100,277 for cl100k_base). This ensures every possible token has exactly one entry.
- Columns: The number of columns is the Embedding Dimension (e.g., 768).

How it works: When you pass a Token ID to the layer, it simply retrieves the row at that index. ID 1 corresponds to Row 1. There is no calculation—just a fast lookup.

---

**Example**

To visualize this, we use a tiny $6 \times 3$ Matrix. This means the model only knows 6 unique tokens, and each token is described by only 3 dimensions. At the start of training, these numbers are random and have no meaning—the model learns the "meaning" through the training.

In [26]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [27]:
# convert token ID 3 to its corresponding embedding vector
# Our Token ID is 3, so we look up the 3rd row of the embedding layer's weight matrix
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [29]:
# Embedding every token
# This transformed matrix now serves as the numerical input for the  subsequent layers of the neural network during training.
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## Encoding word positions

At this stage, we have successfully transformed our Token IDs into dense vectors. However, there is a fundamental problem: **Token Embeddings are position-invariant.**

The embedded vectors of the sentences "The dog bites the man" and "The man bites the dog" are identical in both cases. Without additional information, a Transformer model would treat these sentences as a "bag of words," unaware of the order. To fix this, we use **Positional Embeddings**.

We create a second embedding matrix that doesn't store "word meanings," but rather "position meanings" (e.g., a vector for "Position 0", a vector for "Position 1").

To combine these two pieces of information, we simply add the Token Embedding vector and the Positional Embedding vector together. The resulting vector contains both the semantic meaning of the word and its location in the sequence.

In [45]:
# The BytePair encoder hclk100k_base has a vocabulary size of 10,0277:
# Suppose we want to encode the input tokens into a 256-dimensional vector representation:
vocab_size = 100277
output_dim = 256

# 1.) Initialize the embedding layer with the specified vocabulary size and output dimension
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [55]:
# Extract the first batch of token IDs but with different parameters for the dataloader

max_length = 4
batch_size = 8

dataloader = create_dataloader_v1(
    raw_text, batch_size=batch_size, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)

inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

token_embeddings = token_embedding_layer(inputs)
print("")
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(token_embeddings)

Token IDs:
 tensor([[  697,   346, 20898,    25],
        [ 2360,  7826,   731, 18878],
        [27085,   198, 46847,    25],
        [  220,    16,   272,    13],
        [32620, 19937, 14198, 13465],
        [   11,   220,    16,    14],
        [   17,   272,    13, 60150],
        [  660, 14403,    11,   220]])

Inputs shape:
 torch.Size([8, 4])

torch.Size([8, 4, 256])
tensor([[[ 0.9183,  0.4431,  0.2889,  ..., -1.2531, -0.7126,  0.5632],
         [ 0.3412, -0.4779,  0.7970,  ...,  0.2996,  0.1839,  0.7080],
         [ 0.8796,  0.3248, -0.8520,  ...,  0.3504,  1.6166,  0.1473],
         [ 0.5057, -0.4872, -1.6661,  ...,  0.6667,  0.2808, -0.5583]],

        [[-2.1582,  0.9201, -0.0430,  ..., -1.8297,  0.0038, -1.0591],
         [-0.4789,  0.3494,  0.1615,  ...,  0.4506,  1.2809, -1.3510],
         [ 0.3066,  0.6221, -2.3335,  ..., -0.6599, -1.5638, -0.0299],
         [-0.4559, -1.3369,  0.3483,  ...,  0.0517, -0.6851,  1.0462]],

        [[-0.3369, -0.6294,  0.6345,  ..., -1.3957,

In [56]:
# Extract the first batch of token IDs but with different parameters for the dataloader

max_length = 3
batch_size = 3

dataloader = create_dataloader_v1(
    raw_text, batch_size=batch_size, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)

inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

token_embeddings = token_embedding_layer(inputs)
print("")
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(token_embeddings)

Token IDs:
 tensor([[  697,   346, 20898],
        [   25,  2360,  7826],
        [  731, 18878, 27085]])

Inputs shape:
 torch.Size([3, 3])

torch.Size([3, 3, 256])
tensor([[[ 0.9183,  0.4431,  0.2889,  ..., -1.2531, -0.7126,  0.5632],
         [ 0.3412, -0.4779,  0.7970,  ...,  0.2996,  0.1839,  0.7080],
         [ 0.8796,  0.3248, -0.8520,  ...,  0.3504,  1.6166,  0.1473]],

        [[ 0.5057, -0.4872, -1.6661,  ...,  0.6667,  0.2808, -0.5583],
         [-2.1582,  0.9201, -0.0430,  ..., -1.8297,  0.0038, -1.0591],
         [-0.4789,  0.3494,  0.1615,  ...,  0.4506,  1.2809, -1.3510]],

        [[ 0.3066,  0.6221, -2.3335,  ..., -0.6599, -1.5638, -0.0299],
         [-0.4559, -1.3369,  0.3483,  ...,  0.0517, -0.6851,  1.0462],
         [-0.3369, -0.6294,  0.6345,  ..., -1.3957, -1.6968,  0.0764]]],
       grad_fn=<EmbeddingBackward0>)


In [59]:
# Create a positional embedding layer for the context length of the model
# The context length is the maximum number of tokens that the model can process in a single forward pass.
# Row 0 of the layer stores the vector for: "I am at the 1st position in the sequence."
# Row 1 stores the vector for: "I am at the 2nd position in the sequence"
# and so on...

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[-0.0953, -1.6837, -1.1424, -0.7022, -0.9714,  0.9356,  1.8359,  0.1132,
         -0.8689, -0.0752,  0.1172, -2.1450,  1.6051,  0.6419,  2.0546,  0.1023,
          0.4699, -1.9966, -0.0234, -0.0161, -2.2044, -0.7199, -0.7228, -0.8700,
          1.7233, -0.6316,  2.4492, -1.0824, -1.2053,  0.5678,  2.1723, -0.2593,
          0.1578,  1.1269, -0.4221, -0.2767,  0.8990, -0.2150, -1.1748,  1.8288,
          0.0200, -1.1540,  0.9672,  0.0498,  1.5148,  2.8990,  0.1148,  0.6762,
         -0.8455, -0.0798, -0.3394, -0.9424, -0.5607,  1.8140,  0.3678, -0.3762,
          0.4717, -0.0123,  0.2792, -0.6289,  0.3227,  0.5179, -0.0943,  0.1375,
          0.9851, -0.5210, -0.0477,  0.6403, -0.2154, -1.0666, -0.9786, -0.1159,
          0.7243,  0.0410, -0.5299,  1.6627,  1.0924,  0.9493, -0.1831,  0.1421,
          1.3401,  1.0194,  1.8811,  0.0381, -1.4844, -0.3206, -0.1219,  0.7764,
          1.2554,  1.2812,  2.1154, -1.4395, -0.2878, -0.3383,  0.9855, -0.2153,
      

In [60]:
# Creates a simple tensor of indices: [0, 1, 2, 3]. These are positional addresses.
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(pos_embeddings)

torch.Size([3, 256])
tensor([[-0.0953, -1.6837, -1.1424, -0.7022, -0.9714,  0.9356,  1.8359,  0.1132,
         -0.8689, -0.0752,  0.1172, -2.1450,  1.6051,  0.6419,  2.0546,  0.1023,
          0.4699, -1.9966, -0.0234, -0.0161, -2.2044, -0.7199, -0.7228, -0.8700,
          1.7233, -0.6316,  2.4492, -1.0824, -1.2053,  0.5678,  2.1723, -0.2593,
          0.1578,  1.1269, -0.4221, -0.2767,  0.8990, -0.2150, -1.1748,  1.8288,
          0.0200, -1.1540,  0.9672,  0.0498,  1.5148,  2.8990,  0.1148,  0.6762,
         -0.8455, -0.0798, -0.3394, -0.9424, -0.5607,  1.8140,  0.3678, -0.3762,
          0.4717, -0.0123,  0.2792, -0.6289,  0.3227,  0.5179, -0.0943,  0.1375,
          0.9851, -0.5210, -0.0477,  0.6403, -0.2154, -1.0666, -0.9786, -0.1159,
          0.7243,  0.0410, -0.5299,  1.6627,  1.0924,  0.9493, -0.1831,  0.1421,
          1.3401,  1.0194,  1.8811,  0.0381, -1.4844, -0.3206, -0.1219,  0.7764,
          1.2554,  1.2812,  2.1154, -1.4395, -0.2878, -0.3383,  0.9855, -0.2153,
       

In [61]:
# To create now the final input embeddings used in an LLM, we simply add the token and the positional embeddings:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(input_embeddings)

torch.Size([3, 3, 256])
tensor([[[ 8.2298e-01, -1.2406e+00, -8.5346e-01,  ..., -2.1128e+00,
          -3.8180e+00, -1.0772e+00],
         [-8.9420e-01, -2.4117e-01, -6.4119e-01,  ...,  2.5609e-03,
           1.7971e+00,  1.3955e+00],
         [ 1.4054e+00, -2.8417e-01, -2.6421e+00,  ...,  5.3347e-01,
           3.7535e-01,  5.3207e-01]],

        [[ 4.1037e-01, -2.1709e+00, -2.8085e+00,  ..., -1.9296e-01,
          -2.8246e+00, -2.1988e+00],
         [-3.3936e+00,  1.1569e+00, -1.4813e+00,  ..., -2.1268e+00,
           1.6170e+00, -3.7160e-01],
         [ 4.6930e-02, -2.5961e-01, -1.6286e+00,  ...,  6.3371e-01,
           3.9688e-02, -9.6625e-01]],

        [[ 2.1132e-01, -1.0616e+00, -3.4759e+00,  ..., -1.5195e+00,
          -4.6692e+00, -1.6704e+00],
         [-1.6913e+00, -1.1001e+00, -1.0899e+00,  ..., -2.4541e-01,
           9.2811e-01,  1.7338e+00],
         [ 1.8893e-01, -1.2384e+00, -1.1556e+00,  ..., -1.2126e+00,
          -2.9381e+00,  4.6116e-01]]], grad_fn=<AddBackward0>)


## Short Summary

We have successfully transformed raw text into a input embedding, a format that a Large Language Model can "understand" and process.

To reach the final input, we combined two distinct layers:

- Token Embeddings: Convert Token IDs into 256-dimensional vectors to capture semantic meaning.
- Positional Embeddings: Add unique vectors for each position to capture word order.

The resulting input_embeddings tensor is now ready for the neural network.